In [4]:
import hashlib
import time
import json

class Block:
    def __init__(self, index, timestamp, data, previous_hash):
        self.index = index
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = f"{self.index}{self.timestamp}{self.data}{self.previous_hash}".encode()
        return hashlib.sha256(block_string).hexdigest()

class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        return Block(0, time.time(), "Genesis Block", "0")

    def get_latest_block(self):
        return self.chain[-1]

    def add_block(self, new_block):
        new_block.previous_hash = self.get_latest_block().hash
        new_block.hash = new_block.calculate_hash()
        self.chain.append(new_block)

    def is_chain_valid(self):
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i - 1]
            
            # Verify if hash is correct
            if current_block.hash != current_block.calculate_hash():
                return False
            
            # Verify if block points to previous block's hash
            if current_block.previous_hash != previous_block.hash:
                return False
        return True

    def add_log_entry(self, attributes, success, message, username):
        log_data = {
            "timestamp": time.time(),
            "attributes": attributes,
            "success": success,
            "message": message,
            "username": username
        }
        new_block = Block(len(self.chain), time.time(), log_data, self.get_latest_block().hash)
        self.add_block(new_block)
        print(f"Log entry added to blockchain: {json.dumps(log_data, indent=4)}")

    def verify_access(self, attributes):
        access_granted = False
        for block in self.chain:
            if block.index != 0:  # Skip genesis block
                log_data = block.data
                if log_data["attributes"] == attributes and log_data["success"]:
                    access_granted = True
                    print(f"Access granted for attributes {attributes} on {time.ctime(log_data['timestamp'])}")
                    break
        if not access_granted:
            print("Access denied for specified attributes.")
        return access_granted

    def print_chain(self):
        for block in self.chain:
            print(f"Index: {block.index}")
            print(f"Timestamp: {time.ctime(block.timestamp)}")
            print(f"Data: {block.data}")
            print(f"Hash: {block.hash}")
            print(f"Previous Hash: {block.previous_hash}")
            print("\n")
            
def save_blockchain(blockchain, file_path="blockchain.json"):
    """Save the blockchain to a JSON file."""
    chain_data = []
    for block in blockchain.chain:
        block_dict = {
            "index": block.index,
            "timestamp": block.timestamp,
            "data": block.data,
            "previous_hash": block.previous_hash,
            "hash": block.hash
        }
        chain_data.append(block_dict)
    
    with open(file_path, "w") as file:
        json.dump(chain_data, file, indent=4)
    print(f"Blockchain saved to {file_path}")
    
def load_blockchain(file_path="blockchain.json"):
    """Load the blockchain from a JSON file."""
    with open(file_path, "r") as file:
        chain_data = json.load(file)
    
    blockchain = Blockchain()
    blockchain.chain = []
    
    for block_dict in chain_data:
        block = Block(
            index=block_dict["index"],
            timestamp=block_dict["timestamp"],
            data=block_dict["data"],
            previous_hash=block_dict["previous_hash"]
        )
        block.hash = block_dict["hash"]  # Use the saved hash
        blockchain.chain.append(block)
    
    print(f"Blockchain loaded from {file_path}")
    return blockchain

# Sample usage
if __name__ == "__main__":
    # Initialize blockchain
    log_blockchain = Blockchain()
    # Load blockchain
    #log_blockchain = load_blockchain()

    # Add log entries (simulating decryption attempts)
    log_blockchain.add_log_entry(attributes=["ATTR1", "ATTR2"], success=True, message="Decryption successful.",username="Ramli")
    log_blockchain.add_log_entry(attributes=["ATTR1"], success=False, message="Decryption failed.",username="Ramli")
    log_blockchain.add_log_entry(attributes=["ATTR2", "ATTR3"], success=True, message="Decryption successful.",username="Ramli")
    save_blockchain(log_blockchain)

    # Verify access based on attributes
    log_blockchain.verify_access(attributes=["ATTR1", "ATTR2"])  # Should grant access
    log_blockchain.verify_access(attributes=["ATTR4"])           # Should deny access

    # Print the blockchain to verify structure and integrity
    print("\nBlockchain Structure and Integrity Check:")
    log_blockchain.print_chain()

    # Validate the blockchain's integrity
    is_valid = log_blockchain.is_chain_valid()
    print(f"\nIs blockchain valid? {is_valid}")


Log entry added to blockchain: {
    "timestamp": 1737034392.885661,
    "attributes": [
        "ATTR1",
        "ATTR2"
    ],
    "success": true,
    "message": "Decryption successful.",
    "username": "Ramli"
}
Log entry added to blockchain: {
    "timestamp": 1737034392.885807,
    "attributes": [
        "ATTR1"
    ],
    "success": false,
    "message": "Decryption failed.",
    "username": "Ramli"
}
Log entry added to blockchain: {
    "timestamp": 1737034392.8858979,
    "attributes": [
        "ATTR2",
        "ATTR3"
    ],
    "success": true,
    "message": "Decryption successful.",
    "username": "Ramli"
}
Blockchain saved to blockchain.json
Access granted for attributes ['ATTR1', 'ATTR2'] on Thu Jan 16 19:03:12 2025
Access denied for specified attributes.

Blockchain Structure and Integrity Check:
Index: 0
Timestamp: Thu Jan 16 19:03:12 2025
Data: Genesis Block
Hash: b5d64353f36a261be2b0c51d91f5698447b5820db97381055742a300a979b22c
Previous Hash: 0


Index: 1
Timestam

In [5]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import json
import base64
import json

def save_key(secret_key, file_path="aes_key.key"):
    """Save AES key to a file."""
    with open(file_path, 'wb') as key_file:
        # Encode the key to Base64 and save it
        key_file.write(base64.b64encode(secret_key))


def save_encrypted_file(ciphertext, metadata, file_path="encrypted_data.json"):
    """Save encrypted data and metadata to a file."""
    data_to_save = {
        "ciphertext": ciphertext,
        "metadata": metadata
    }
    with open(file_path, 'w') as file:
        json.dump(data_to_save, file)

def load_encrypted_file(file_path="encrypted_data.json"):
    """Load encrypted data and metadata from a file."""
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data["ciphertext"], data["metadata"]
def load_key(file_path="aes_key.key"):
    """Load AES key from a file."""
    with open(file_path, 'rb') as key_file:
        # Decode the Base64 encoded key
        return base64.b64decode(key_file.read())
def encrypt_data(data, policy_attributes, secret_key):
    """Encrypt data with a policy."""
    cipher = AES.new(secret_key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(data.encode())
    
    # Store policy in metadata
    metadata = {
        'policy': policy_attributes,
        'nonce': cipher.nonce.hex(),
        'tag': tag.hex()
    }
    return ciphertext.hex(), metadata

def decrypt_data(ciphertext, metadata, user_attributes, secret_key):
    """Decrypt data if user satisfies the policy."""
    required_attributes = set(metadata['policy'])
    user_attributes_set = set(user_attributes)

    # Check if user satisfies policy
    if not required_attributes.issubset(user_attributes_set):
        raise PermissionError("User does not satisfy the policy!")
    
    cipher = AES.new(secret_key, AES.MODE_GCM, nonce=bytes.fromhex(metadata['nonce']))
    plaintext = cipher.decrypt_and_verify(bytes.fromhex(ciphertext), bytes.fromhex(metadata['tag']))
    return plaintext.decode()

# Example usage
log_blockchain = load_blockchain()
secret_key = get_random_bytes(16)  
save_key(secret_key)
data = "Confidential data"
# Load the AES key when needed
loaded_key = load_key()
policy = ['attr1', 'attr2']
# Encrypt data using the loaded key
ciphertext, metadata = encrypt_data(data, policy, loaded_key)
# Save the encrypted data and metadata to a file
save_encrypted_file(ciphertext, metadata)
user_attributes = ['attr1', 'attr2', 'attr3']
log_blockchain.add_log_entry(attributes=user_attributes, success=True, message="Decryption successful.",username="Ramli")
loaded_ciphertext, loaded_metadata = load_encrypted_file()
# Decrypt data using the loaded key
try:
    decrypted_data = decrypt_data(loaded_ciphertext, loaded_metadata, user_attributes, loaded_key)
    print("Decrypted Data:", decrypted_data)
except PermissionError as e:
    print(str(e))



Blockchain loaded from blockchain.json
Log entry added to blockchain: {
    "timestamp": 1737034392.9033659,
    "attributes": [
        "attr1",
        "attr2",
        "attr3"
    ],
    "success": true,
    "message": "Decryption successful.",
    "username": "Ramli"
}
Decrypted Data: Confidential data
